# 19.3 Capstone — Inventory Service on SQLite

**Prerequisites:** 05 OOPs (5.3 dataclasses), 06 Exception Handling (6.2 custom exceptions, 6.3 context managers), 10 Database (10.1 SQL, 10.3 SQLite), 15 Testing and Debugging (15.3–15.4 pytest and fixtures), 16 Type Hints (16.4 Protocols)  
**Target:** Python 3.12+

### What you'll build

A warehouse inventory service — products, stock levels and an audit trail of every
movement — built end to end:

- A **schema** where the movement ledger is the source of truth, with `CHECK`
  constraints, enforced foreign keys and `STRICT` tables
- A **data-access layer** (`InventoryRepo`): injected connection, parameterised
  queries only, frozen dataclasses out, domain exceptions up
- **Transactions** doing the heavy lifting — multi-line orders that ship
  completely or not at all, and an idempotent `receive`
- The **queries** a warehouse actually asks: stock on hand, low-stock, history
- A **`Protocol` seam** so service logic never learns SQL exists — verified by
  mypy on files extracted from this very notebook
- A **pytest suite** on `:memory:` databases proving every invariant

Nothing here touches your disk except one temporary scratch directory, removed
in the final cell.

## The problem

A small warehouse. Deliveries arrive on pallets, orders ship in cardboard,
someone occasionally drops a box. The software's one unforgivable failure is
**losing a unit** — stock the system swears exists but the shelf disproves, or
the reverse. Miscounted money is annoying; miscounted inventory is
embarrassing, because anyone can go and *look*.

Three invariants define "correct", and every design decision below serves one
of them:

1. **Stock never goes negative.** You cannot ship what is not on the shelf.
2. **Every change leaves an audit row.** If a number moved, there is a row
   saying what, how much and why — no write path skips the ledger.
3. **Multi-step operations are all-or-nothing.** An order of three lines must
   never half-happen, whatever fails in the middle — and operations that
   interleave must not corrupt the totals.

The plan of attack:

| Layer | Tools | Folder |
|---|---|---|
| Schema: constraints + a movement ledger | SQL, `CHECK`, foreign keys, `STRICT` | **10.1**, **10.3** |
| Data access: `InventoryRepo` | `sqlite3`, placeholders, `with conn:` | **10.3**, **6.3** |
| Rows and errors | frozen dataclasses, custom exceptions | **5.3**, **6.2** |
| The seam above the repo | `Protocol` + mypy in a subprocess | **16.4**, **4.5** |
| Proof | pytest on `:memory:` databases | **15.3–15.4**, **10.3** |

---

## 1. The schema — a ledger, not a number

The tempting design stores stock as a column and updates it in place. Compare:

| | `products.on_hand` column | `stock_movements` ledger (chosen) |
|---|---|---|
| read current stock | free — `SELECT on_hand` | `SUM(quantity)` over the ledger |
| write | `UPDATE ... SET on_hand = on_hand - 5` | `INSERT` one row |
| audit trail | gone — overwritten every time | **is** the table |
| a bug double-ships | silently wrong forever | recomputable, provable |
| invariant 2 | needs discipline in every writer | structurally guaranteed |

We choose the ledger: **stock on hand is `SUM(quantity)` — always derived,
never stored.** Reads cost a `GROUP BY`; in exchange, invariant 2 cannot be
violated by a forgetful writer, and any cached total can be *audited* against
the movements (the reconciliation query, below). Banks, accountants and
warehouse systems all land on this shape: movements are facts, totals are
opinions derived from them.

Three pieces of schema armour from **10.3**:

- **`CHECK` constraints** — a movement of `0`, a `'receive'` that subtracts, or
  a kind outside `('receive', 'ship', 'adjust')` is rejected by the *database*,
  whatever bugs the Python layer grows.
- **`PRAGMA foreign_keys = ON`** — 🔴 off by default, and per *connection*;
  without it SQLite happily books movements against products that do not exist
  (the orphan-row lesson).
- **`STRICT` tables** (SQLite 3.37+, 2021) — without `STRICT`, SQLite lets
  `'twelve'` into an `INTEGER` column. ⚠️ Feature-gate on
  `sqlite3.sqlite_version_info` — the **tuple** — never the string, where
  `"3.9.0" > "3.37.0"`.

In [ ]:
# [inventory.py] -- schema and connection helpers
# Cells whose first line carries this tag are later extracted, verbatim, into
# a real module file -- see "From notebook to module" below.
from __future__ import annotations

import sqlite3
from collections.abc import Mapping
from dataclasses import dataclass
from typing import Any

# STRICT tables arrived in SQLite 3.37 (2021-11).
# ⚠️ Gate on the version TUPLE -- "3.9.0" > "3.37.0" as strings (10.3).
STRICT_TABLES: bool = sqlite3.sqlite_version_info >= (3, 37, 0)
_STRICT = "STRICT" if STRICT_TABLES else ""

SCHEMA = f"""
CREATE TABLE IF NOT EXISTS products (
    id            INTEGER PRIMARY KEY,
    sku           TEXT    NOT NULL UNIQUE CHECK (length(sku) > 0),
    name          TEXT    NOT NULL CHECK (length(name) > 0),
    reorder_level INTEGER NOT NULL DEFAULT 0 CHECK (reorder_level >= 0)
) {_STRICT};

CREATE TABLE IF NOT EXISTS stock_movements (
    id           INTEGER PRIMARY KEY,
    product_id   INTEGER NOT NULL REFERENCES products(id),
    kind         TEXT    NOT NULL CHECK (kind IN ('receive', 'ship', 'adjust')),
    quantity     INTEGER NOT NULL,
    reason       TEXT,
    client_token TEXT    UNIQUE,
    created_at   TEXT    NOT NULL DEFAULT (datetime('now')),
    CHECK (
           (kind = 'receive' AND quantity > 0)
        OR (kind = 'ship'    AND quantity < 0)
        OR (kind = 'adjust'  AND quantity != 0)
    )
) {_STRICT};

CREATE INDEX IF NOT EXISTS idx_movements_product ON stock_movements (product_id);

CREATE VIEW IF NOT EXISTS v_stock_on_hand AS
    SELECT p.id AS product_id, p.sku, COALESCE(SUM(m.quantity), 0) AS on_hand
    FROM products p
    LEFT JOIN stock_movements m ON m.product_id = p.id
    GROUP BY p.id;
"""


def connect(path: str = ":memory:") -> sqlite3.Connection:
    """Open a connection with the settings EVERY connection needs (10.3)."""
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row              # rows by column name
    conn.execute("PRAGMA foreign_keys = ON")    # 🔴 off by default, per connection
    return conn


def init_db(conn: sqlite3.Connection) -> None:
    """Create the schema. IF NOT EXISTS makes a second call harmless."""
    conn.executescript(SCHEMA)

In [ ]:
demo = connect()                    # ':memory:' -- nothing on disk
init_db(demo)

print(f"SQLite {sqlite3.sqlite_version}  |  "
      f"STRICT tables: {'on' if STRICT_TABLES else 'off (needs 3.37+)'}")
print("foreign_keys:", demo.execute("PRAGMA foreign_keys").fetchone()[0], "(1 = on)")

objects = demo.execute(
    "SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY 1, 2"
).fetchall()
print("schema objects:", ", ".join(f"{r['name']} ({r['type']})" for r in objects))

# One legitimate product, so the rejection demos have something to reference
demo.execute("INSERT INTO products (sku, name, reorder_level) "
             "VALUES ('CBL-3M', '3 m HDMI cable', 10)")
demo.commit()

# ---- The schema defends itself ----
attempts = [
    ("orphan movement - product 999 does not exist (FK)",
     "INSERT INTO stock_movements (product_id, kind, quantity) VALUES (999, 'receive', 5)"),
    ("kind outside the enum (CHECK)",
     "INSERT INTO stock_movements (product_id, kind, quantity) VALUES (1, 'audit', 5)"),
    ("a 'receive' that removes stock (CHECK)",
     "INSERT INTO stock_movements (product_id, kind, quantity) VALUES (1, 'receive', -5)"),
]
if STRICT_TABLES:
    attempts.append(
        ("TEXT sneaking into an INTEGER column (STRICT)",
         "INSERT INTO stock_movements (product_id, kind, quantity) VALUES (1, 'receive', 'twelve')"))

print()
for label, sql in attempts:
    try:
        demo.execute(sql)
    except sqlite3.Error as exc:
        print(f"  rejected: {label}")
        print(f"            {type(exc).__name__}: {exc}")

Every rejection came from *inside the database*, before any Python `if` had a
say — the last line of defence for the day the application layer grows a bug.

### The reconciliation query

Choosing the ledger does not ban caching — it means any cached total is
*checkable*. The `v_stock_on_hand` view computes truth from movements; if a
denormalised copy ever exists (for speed, or in a legacy table), one `JOIN`
convicts it the moment it drifts. Watch a forgetful writer get caught:

In [ ]:
# ---- Seed the ledger with plain SQL (the repo arrives next section) ----
with demo:
    demo.execute("INSERT INTO stock_movements (product_id, kind, quantity, reason) "
                 "VALUES (1, 'receive', 40, 'PO-778')")
    demo.execute("INSERT INTO stock_movements (product_id, kind, quantity, reason) "
                 "VALUES (1, 'ship', -5, 'order SO-900')")

row = demo.execute("SELECT on_hand FROM v_stock_on_hand WHERE sku = 'CBL-3M'").fetchone()
print("ledger says CBL-3M on hand:", row["on_hand"])

# ---- The design we did NOT choose, and its failure mode ----
with demo:
    demo.execute("DROP TABLE IF EXISTS stock_cache")
    demo.execute("CREATE TABLE stock_cache "
                 "(product_id INTEGER PRIMARY KEY, cached_on_hand INTEGER NOT NULL)")
    demo.execute("INSERT INTO stock_cache SELECT product_id, on_hand FROM v_stock_on_hand")

# A later writer updates the ledger but -- the bug -- forgets the cache:
with demo:
    demo.execute("INSERT INTO stock_movements (product_id, kind, quantity, reason) "
                 "VALUES (1, 'ship', -5, 'order SO-901')")

drift = demo.execute("""
    SELECT v.sku, c.cached_on_hand, v.on_hand AS ledger_on_hand
    FROM stock_cache c
    JOIN v_stock_on_hand v ON v.product_id = c.product_id
    WHERE c.cached_on_hand != v.on_hand
""").fetchall()
for r in drift:
    print(f"drift caught on {r['sku']}: cache says {r['cached_on_hand']}, "
          f"the ledger proves {r['ledger_on_hand']}")

print("\nthe reconciliation query is the audit; the ledger is the truth")
demo.close()

---

## 2. The data-access layer

One class owns every line of SQL: `InventoryRepo`. The rules it lives by:

- **The connection is injected.** The repo never calls `sqlite3.connect` —
  production hands in a file-backed database, every test hands in `:memory:`
  (**10.3**). Swapping storage is a constructor argument, not a rewrite.
- **Placeholders only.** Any f-string next to `execute()` is a bug (**10.3**);
  the one deliberate exception below exists to be attacked.
- **Domain exceptions up** (**6.2**) — callers catch `InsufficientStockError`
  and read `.sku`, `.requested`, `.available`; nobody parses `sqlite3` message
  strings.
- **Frozen dataclasses out** (**5.3**) — queries return `Product` and
  `Movement` values, not tuples the caller must index blind.

First the exceptions, then the row types, then the repo itself.

In [ ]:
# [inventory.py] -- domain exceptions (6.2)

class InventoryError(Exception):
    """Base class for everything this service raises. Catch this to catch all."""


class UnknownProductError(InventoryError):
    """A SKU that is not in the catalogue."""


class DuplicateProductError(InventoryError):
    """A SKU that is already in the catalogue."""


class InsufficientStockError(InventoryError):
    """Asked to take out more stock than the ledger holds.

    Carries the numbers as attributes (6.2): callers can render a message,
    retry with less, or trigger a re-order -- without parsing a string.
    """

    def __init__(self, sku: str, requested: int, available: int) -> None:
        self.sku = sku
        self.requested = requested
        self.available = available
        super().__init__(f"cannot take {requested} x {sku}: only {available} on hand")

In [ ]:
# [inventory.py] -- row types (5.3): frozen, slotted, hashable

@dataclass(frozen=True, slots=True)
class Product:
    id: int
    sku: str
    name: str
    reorder_level: int


@dataclass(frozen=True, slots=True)
class Movement:
    id: int
    product_id: int
    kind: str            # 'receive' | 'ship' | 'adjust'
    quantity: int        # signed: +receive, -ship, +/-adjust
    reason: str | None
    client_token: str | None
    created_at: str      # UTC, 'YYYY-MM-DD HH:MM:SS'


def _movement_row(cursor: sqlite3.Cursor, row: tuple[Any, ...]) -> Movement:
    """A cursor-level row_factory (10.3): SELECTed rows arrive as Movements."""
    return Movement(*row)

### Who owns the transaction?

One rule keeps the repo honest: **one public method = one transaction.**
Public methods open `with self.conn:` (**6.3**); private helpers like
`_record` never do.

🔴 The trap this avoids: `sqlite3`'s `with conn:` does not nest. An inner
block's exit calls `commit()` — committing the *outer* method's half-finished
work and quietly destroying its all-or-nothing guarantee. If a helper manages
the transaction, `ship_order`'s rollback has nothing left to roll back.

⚠️ And remember (**10.3**): `with conn:` manages the *transaction*, not the
connection — it never calls `close()`.

In [ ]:
# [inventory.py] -- the data-access layer

class InventoryRepo:
    """All SQL for the inventory lives here; nothing above this layer writes SQL.

    The connection is INJECTED, not created here: production passes a
    file-backed database, every test passes ':memory:' (10.3). One public
    method = one transaction = one unit of work. (⚠️ nuance: sqlite3's legacy
    transaction control defers BEGIN until the first write — see Common Mistakes.)
    """

    def __init__(self, conn: sqlite3.Connection) -> None:
        self.conn = conn

    # -- products ----------------------------------------------------------
    def add_product(self, sku: str, name: str, *, reorder_level: int = 0) -> Product:
        try:
            with self.conn:
                cur = self.conn.execute(
                    "INSERT INTO products (sku, name, reorder_level) VALUES (?, ?, ?)",
                    (sku, name, reorder_level),
                )
        except sqlite3.IntegrityError as exc:
            raise DuplicateProductError(f"SKU {sku!r} already exists") from exc  # 6.2
        assert cur.lastrowid is not None
        return Product(cur.lastrowid, sku, name, reorder_level)

    def get_product(self, sku: str) -> Product:
        row = self.conn.execute(
            "SELECT id, sku, name, reorder_level FROM products WHERE sku = ?", (sku,)
        ).fetchone()
        if row is None:
            raise UnknownProductError(f"no product with SKU {sku!r}")
        return Product(row["id"], row["sku"], row["name"], row["reorder_level"])

    def products(self) -> list[Product]:
        cur = self.conn.execute(
            "SELECT id, sku, name, reorder_level FROM products ORDER BY sku")
        return [Product(r["id"], r["sku"], r["name"], r["reorder_level"]) for r in cur]

    # -- writing stock -----------------------------------------------------
    def receive(self, sku: str, quantity: int, *, reason: str | None = None,
                client_token: str | None = None) -> Movement:
        """Book incoming stock. With a client_token, safe to call twice (18.3)."""
        if quantity <= 0:
            raise ValueError(f"receive quantity must be positive, got {quantity}")
        with self.conn:
            product = self.get_product(sku)
            if client_token is not None:
                row = self.conn.execute(
                    "SELECT id FROM stock_movements WHERE client_token = ?",
                    (client_token,),
                ).fetchone()
                if row is not None:              # replay -> the ORIGINAL movement
                    return self._movement(row["id"])
            return self._record(product.id, "receive", quantity,
                                reason=reason, client_token=client_token)

    def ship_order(self, order_ref: str, items: Mapping[str, int]) -> list[Movement]:
        """Ship every line of an order, or nothing at all.

        `with self.conn:` (6.3) makes the whole loop ONE transaction: commit on
        success, ROLLBACK on any exception (10.3) -- including our own
        InsufficientStockError raised halfway through the lines.
        """
        movements: list[Movement] = []
        with self.conn:
            for sku, quantity in items.items():
                if quantity <= 0:
                    raise ValueError(f"ship quantity must be positive, got {quantity}")
                product = self.get_product(sku)
                available = self._on_hand(product.id)
                if available < quantity:                     # invariant 1
                    raise InsufficientStockError(sku, quantity, available)
                movements.append(
                    self._record(product.id, "ship", -quantity,
                                 reason=f"order {order_ref}"))
        return movements

    def adjust(self, sku: str, delta: int, *, reason: str) -> Movement:
        """Manual correction (damage, recount). The reason is NOT optional."""
        if delta == 0:
            raise ValueError("an adjustment of 0 would record nothing")
        with self.conn:
            product = self.get_product(sku)
            available = self._on_hand(product.id)
            if available + delta < 0:                        # invariant 1
                raise InsufficientStockError(sku, -delta, available)
            return self._record(product.id, "adjust", delta, reason=reason)

    # -- reading -----------------------------------------------------------
    def stock_level(self, sku: str) -> int:
        """Current on-hand = SUM over the ledger. No cached column to drift."""
        return self._on_hand(self.get_product(sku).id)

    def history(self, sku: str) -> list[Movement]:
        self.get_product(sku)                    # unknown SKU -> raise, not []
        cur = self.conn.execute(
            """
            SELECT m.id, m.product_id, m.kind, m.quantity, m.reason,
                   m.client_token, m.created_at
            FROM stock_movements m
            JOIN products p ON p.id = m.product_id
            WHERE p.sku = ?
            ORDER BY m.id
            """,
            (sku,),
        )
        cur.row_factory = _movement_row          # rows arrive as Movement (5.3)
        return cur.fetchall()

    def stock_report(self) -> list[sqlite3.Row]:
        """On-hand per product: one GROUP BY over the ledger (10.1)."""
        return self.conn.execute(
            """
            SELECT p.sku, p.name, p.reorder_level,
                   COALESCE(SUM(m.quantity), 0) AS on_hand
            FROM products p
            LEFT JOIN stock_movements m ON m.product_id = p.id
            GROUP BY p.id
            ORDER BY p.sku
            """
        ).fetchall()

    def low_stock(self) -> list[sqlite3.Row]:
        """Products below their reorder level -- GROUP BY + HAVING (10.1)."""
        return self.conn.execute(
            """
            SELECT p.sku, p.name, p.reorder_level,
                   COALESCE(SUM(m.quantity), 0) AS on_hand
            FROM products p
            LEFT JOIN stock_movements m ON m.product_id = p.id
            GROUP BY p.id
            HAVING on_hand < p.reorder_level
            ORDER BY p.sku
            """
        ).fetchall()

    # -- internals ---------------------------------------------------------
    def _on_hand(self, product_id: int) -> int:
        row = self.conn.execute(
            "SELECT COALESCE(SUM(quantity), 0) AS on_hand "
            "FROM stock_movements WHERE product_id = ?",
            (product_id,),
        ).fetchone()
        return int(row["on_hand"])

    def _record(self, product_id: int, kind: str, quantity: int, *,
                reason: str | None, client_token: str | None = None) -> Movement:
        # ⚠️ No `with self.conn:` here. The PUBLIC method owns the transaction;
        # a nested `with conn:` would COMMIT on exit and silently break
        # ship_order's all-or-nothing guarantee.
        cur = self.conn.execute(
            "INSERT INTO stock_movements (product_id, kind, quantity, reason, client_token) "
            "VALUES (?, ?, ?, ?, ?)",
            (product_id, kind, quantity, reason, client_token),
        )
        assert cur.lastrowid is not None
        return self._movement(cur.lastrowid)

    def _movement(self, movement_id: int) -> Movement:
        cur = self.conn.execute(
            "SELECT id, product_id, kind, quantity, reason, client_token, created_at "
            "FROM stock_movements WHERE id = ?",
            (movement_id,),
        )
        cur.row_factory = _movement_row
        movement: Movement | None = cur.fetchone()
        assert movement is not None
        return movement

In [ ]:
conn = connect()                     # ':memory:' -- the whole warehouse in RAM
init_db(conn)
repo = InventoryRepo(conn)

for sku, name, level in [
    ("CBL-3M", "3 m HDMI cable", 10),
    ("SSD-1TB", "1 TB NVMe drive", 4),
    ("KBD-TKL", "Tenkeyless keyboard", 5),
]:
    repo.add_product(sku, name, reorder_level=level)

repo.receive("CBL-3M", 40, reason="purchase order PO-778")
repo.receive("SSD-1TB", 12, reason="purchase order PO-778")
repo.receive("KBD-TKL", 6, reason="purchase order PO-779")


def levels(repo: InventoryRepo) -> dict[str, int]:
    """{sku: on_hand} -- the whole warehouse at a glance."""
    return {row["sku"]: row["on_hand"] for row in repo.stock_report()}


def movement_count(repo: InventoryRepo) -> int:
    return int(repo.conn.execute(
        "SELECT COUNT(*) AS n FROM stock_movements").fetchone()["n"])


print("after receiving: ", levels(repo))

shipped = repo.ship_order("SO-1001", {"CBL-3M": 5, "KBD-TKL": 2})
print("order SO-1001:   ",
      ", ".join(f"#{m.id} {m.kind} {m.quantity:+}" for m in shipped))
print("after shipping:  ", levels(repo))

# The domain exception carries DATA, not just a message (6.2)
try:
    repo.ship_order("SO-1002", {"SSD-1TB": 50})
except InsufficientStockError as exc:
    print("\nrefused:", exc)
    print(f"  as data -> sku={exc.sku!r}  requested={exc.requested}  "
          f"available={exc.available}")

print("\naudit rows so far:", movement_count(repo))

### 🔴 The attack this layer makes impossible

Every repo query uses `?` placeholders. Here is what that discipline is worth —
the **10.3** injection demo, replayed against this schema. The unsafe function
below is the only f-string SQL in this notebook, and it exists to be attacked.

In [ ]:
def find_product_unsafe(conn: sqlite3.Connection, sku: str) -> list[tuple[str, str]]:
    """🔴 NEVER write this -- the value is PASTED INTO the SQL (10.3)."""
    sql = f"SELECT sku, name FROM products WHERE sku = '{sku}'"
    print("  SQL sent:", sql)
    return [(r["sku"], r["name"]) for r in conn.execute(sql)]


print("legitimate call:")
print("  ->", find_product_unsafe(conn, "CBL-3M"))

attack = "' OR '1'='1"
print("\n🔴 the same function, attacked:")
print("  ->", find_product_unsafe(conn, attack))
print("  ^ the whole catalogue leaked -- the input became SQL")

print("\n✅ the repo, attacked the same way:")
try:
    repo.get_product(attack)
except UnknownProductError as exc:
    print("  ->", exc)
    print("  ^ the ? placeholder searched for that text LITERALLY -- no such SKU")

---

## 3. Transactions are the star

Order `SO-2000` wants 5 cables and 40 SSDs. There are only 12 SSDs. The correct
outcome is obvious: *refuse the whole order, change nothing.*

But shipping is a loop — check a line, insert a movement, check the next line…
If line 1 is **committed** before line 2 fails, the warehouse has half-shipped
an order: cables deducted, order unfulfillable, invariant 3 in pieces.

First, the broken version every codebase contains at some point: a transaction
per line.

In [ ]:
def ship_order_unsafe(conn: sqlite3.Connection, order_ref: str,
                      items: Mapping[str, int]) -> None:
    """🔴 One transaction PER LINE. Looks fine until a later line fails."""
    for sku, quantity in items.items():
        row = conn.execute(
            "SELECT product_id, on_hand FROM v_stock_on_hand WHERE sku = ?",
            (sku,)).fetchone()
        if row["on_hand"] < quantity:
            raise InsufficientStockError(sku, quantity, row["on_hand"])
        with conn:                            # 🔴 commits THIS line immediately
            conn.execute(
                "INSERT INTO stock_movements (product_id, kind, quantity, reason) "
                "VALUES (?, 'ship', ?, ?)",
                (row["product_id"], -quantity, f"order {order_ref}"))


print("before:  ", levels(repo), "|", movement_count(repo), "audit rows")

try:
    ship_order_unsafe(conn, "SO-2000", {"CBL-3M": 5, "SSD-1TB": 40})
except InsufficientStockError as exc:
    print("line 2 failed:", exc)

print("after:   ", levels(repo), "|", movement_count(repo), "audit rows")
print("🔴 line 1 shipped AND committed -- order SO-2000 is half-fulfilled forever")

# ---- Repairing a ledger: with a new movement, never an UPDATE ----
repo.adjust("CBL-3M", +5, reason="reversal: order SO-2000 failed after line 1")
print("\nrepaired:", levels(repo), "|", movement_count(repo), "audit rows")
print("the mistake AND its correction are both on the record")

Line 1 committed; line 2 threw; nobody cleaned up. And note the repair: on a
ledger you fix mistakes with a **new movement** — an `adjust`, reason attached —
never by editing history. The mistake and its correction are both auditable.

### `with conn:` makes the loop one atomic unit

`InventoryRepo.ship_order` wraps the whole loop in a single `with self.conn:`
(**6.3**): commit if the block finishes, **rollback if anything raises**
(**10.3**). The `INSERT` for line 1 genuinely executes — and when line 2 raises
`InsufficientStockError`, that insert is un-happened along with everything
else. Same doomed order, same code path, zero damage:

In [ ]:
before_levels, before_count = levels(repo), movement_count(repo)
print("before:        ", before_levels, "|", before_count, "audit rows")

try:
    repo.ship_order("SO-2001", {"CBL-3M": 5, "SSD-1TB": 40})   # same doomed order
except InsufficientStockError as exc:
    print("refused:       ", exc)

print("after failure: ", levels(repo), "|", movement_count(repo), "audit rows")
assert levels(repo) == before_levels and movement_count(repo) == before_count
print("✅ line 1's INSERT ran inside the transaction -- and rolled back with it")

# A fulfillable order sails through the very same code path
fulfilled = repo.ship_order("SO-2002", {"CBL-3M": 5, "SSD-1TB": 4})
print(f"\nSO-2002 shipped, {len(fulfilled)} lines in ONE transaction")
print("after success: ", levels(repo), "|", movement_count(repo), "audit rows")

### An idempotent `receive` — 18.3's lesson, in SQL

Deliveries are booked by systems that retry: a supplier webhook fires twice, a
warehouse scanner resends on timeout. `receive("SSD-1TB", 10)` called twice
books **20** units — the classic double-`POST` from **18.3**, where the fix was
an `Idempotency-Key` header.

The SQL translation: the caller names the physical event with a **client
token** (`"PO-990/delivery-1"`); the `client_token` column is `UNIQUE`; a
replay finds the existing movement and returns it instead of inserting. Same
token → same movement → stock counted once, however many times the message
arrives.

In [ ]:
token = "PO-990/delivery-1"          # one physical delivery, one token (18.3)

first = repo.receive("SSD-1TB", 10, reason="purchase order PO-990",
                     client_token=token)
print("first call: movement", first.id, "| SSD-1TB =", repo.stock_level("SSD-1TB"))

replay = repo.receive("SSD-1TB", 10, reason="purchase order PO-990",
                      client_token=token)
print("replay:     movement", replay.id, "| SSD-1TB =", repo.stock_level("SSD-1TB"))

assert first.id == replay.id
print("✅ same movement returned, stock counted ONCE --",
      movement_count(repo), "audit rows")

# And because client_token is UNIQUE, even two processes racing on the same
# token cannot double-book: the second INSERT dies on the constraint.

---

## 4. Queries that answer real questions

The ledger design pays rent at read time (**10.1**): stock per product is a
`GROUP BY`, the low-stock report is the same query plus `HAVING`, and a
product's life story is a `JOIN` ordered by movement id.

In [ ]:
print(f"{'SKU':<9} {'product':<21} {'on hand':>8} {'reorder at':>11}")
for row in repo.stock_report():
    print(f"{row['sku']:<9} {row['name']:<21} {row['on_hand']:>8} "
          f"{row['reorder_level']:>11}")

print("\nlow stock (HAVING on_hand < reorder_level):")
for row in repo.low_stock():
    print(f"  reorder {row['sku']}: {row['on_hand']} on hand, "
          f"threshold {row['reorder_level']}")

In [ ]:
print("the life of CBL-3M, straight from the audit trail:")
for m in repo.history("CBL-3M"):
    print(f"  #{m.id:<3} {m.kind:<8} {m.quantity:+4}   {m.reason}")

print("\nreorder margin via the view (10.1: JOIN against v_stock_on_hand):")
rows = conn.execute("""
    SELECT p.sku, v.on_hand, p.reorder_level,
           v.on_hand - p.reorder_level AS margin
    FROM v_stock_on_hand v
    JOIN products p ON p.id = v.product_id
    ORDER BY margin
""").fetchall()
for r in rows:
    flag = "   <- reorder" if r["margin"] < 0 else ""
    print(f"  {r['sku']:<9} on_hand={r['on_hand']:<4} margin={r['margin']:+4}{flag}")

---

## 5. The seam — a `Protocol` above the repo (16.4)

Service-level logic ("which SKUs need reordering?") needs stock numbers, not
SQL. Give it a **`Protocol`**: the service declares the shape it needs, and
anything with that shape qualifies — `InventoryRepo` on SQLite today, a
Postgres repo next year, a dict-backed fake in every unit test.

The point of structural typing here: `InventoryRepo` never imports
`StockStore`, never inherits from it, never registers — **the shape alone is
the contract**, and mypy verifies it without the two classes ever meeting at
runtime.

In [ ]:
# [stock_store.py] -- the seam between service logic and storage (16.4)
from __future__ import annotations

from collections.abc import Iterable
from typing import TYPE_CHECKING, Protocol

if TYPE_CHECKING:                    # import needed only for the static proof
    from inventory import InventoryRepo


class StockStore(Protocol):
    """What the service layer needs. Note what is absent: anything about SQL."""

    def stock_level(self, sku: str) -> int: ...


def restock_report(store: StockStore, catalogue: Iterable[tuple[str, int]]) -> list[str]:
    """SKUs that need reordering -- against ANY StockStore."""
    return [sku for sku, reorder_level in catalogue
            if store.stock_level(sku) < reorder_level]


class InMemoryStockStore:
    """The whole fake. It has never heard of StockStore -- and satisfies it."""

    def __init__(self, levels: dict[str, int]) -> None:
        self.levels = levels

    def stock_level(self, sku: str) -> int:
        return self.levels.get(sku, 0)


def _static_proof(repo: InventoryRepo, fake: InMemoryStockStore) -> None:
    """For mypy only: both implementations are assignable to the Protocol."""
    stores: list[StockStore] = [repo, fake]

In [ ]:
catalogue = [(p.sku, p.reorder_level) for p in repo.products()]

db_answer = restock_report(repo, catalogue)         # SQLite behind the seam
fake = InMemoryStockStore(levels(repo))             # a dict behind the seam
fake_answer = restock_report(fake, catalogue)

print("catalogue:           ", catalogue)
print("repo (SQLite) says:  ", db_answer)
print("in-memory fake says: ", fake_answer)
assert db_answer == fake_answer == ["KBD-TKL"]
print("✅ the service function cannot tell the difference -- that IS the seam")

### From notebook to module — then let mypy at it

The tagged cells above (`# [inventory.py]`, `# [stock_store.py]`) now become
real files: the next cell reads this notebook's own `.ipynb` JSON, concatenates
the tagged cells verbatim, and writes them into a temp project. No copy to
drift out of date, because there is no copy — the notebook is the single
source.

⚠️ It reads the notebook file *on disk*, i.e. the last **saved** state — save
before re-running if you edit a tagged cell.

Then `mypy --strict` type-checks the extracted files in a subprocess, exactly
as **4.5** and **16.4** did — including the structural proof that both
`InventoryRepo` and the five-line fake satisfy `StockStore`.

In [ ]:
import json
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py193_"))     # the ONLY thing on disk
PROJ = WORK / "inventory_service"
PROJ.mkdir()

NOTEBOOK = Path("19.3 Inventory Service on SQLite.ipynb")


def extract_module(marker: str, dest: Path) -> str:
    """Concatenate the code cells tagged '# [marker]' into a real module file."""
    cells = json.loads(NOTEBOOK.read_text(encoding="utf-8"))["cells"]
    parts = ["".join(c["source"]) for c in cells
             if c["cell_type"] == "code"
             and "".join(c["source"]).startswith(f"# [{marker}]")]
    if not parts:
        raise RuntimeError(f"no cells tagged '# [{marker}]' -- notebook renamed?")
    dest.write_text("\n\n\n".join(parts) + "\n", encoding="utf-8")
    lines = dest.read_text(encoding="utf-8").count("\n")
    return f"{dest.name:<15} <- {len(parts)} tagged cell(s), {lines} lines"


print(extract_module("inventory.py", PROJ / "inventory.py"))
print(extract_module("stock_store.py", PROJ / "stock_store.py"))


def mypy(*files: str) -> str:
    """Type-check files with mypy in a subprocess (4.5 / 16.4 pattern)."""
    done = subprocess.run(
        [sys.executable, "-m", "mypy", "--strict", *files,
         "--cache-dir", str(WORK / ".mypy_cache"), "--no-color-output"],
        cwd=PROJ, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip()
    return (f"$ mypy --strict {' '.join(files)}\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print()
print(mypy("inventory.py", "stock_store.py"))

In [ ]:
# ---- What a WRONG implementation looks like to the checker ----
(PROJ / "broken_store.py").write_text(textwrap.dedent("""
    from stock_store import StockStore


    class CountingStore:
        # Looks close -- but stock_level answers with a string.

        def __init__(self) -> None:
            self.seen: list[str] = []

        def stock_level(self, sku: str) -> str:      # int expected
            self.seen.append(sku)
            return "12"


    def wire_up(store: CountingStore) -> StockStore:
        return store
""").lstrip(), encoding="utf-8")

print(mypy("broken_store.py"))

---

## 6. The test suite — invariants, executed

Every test gets a **fresh `:memory:` database** from a fixture (**10.3**,
**15.4**): total isolation, no files, microseconds each. The suite is the three
invariants plus this section's guarantees, written as executable claims:

| Test | Pins down |
|---|---|
| `test_new_product_has_zero_stock` | `SUM` over an empty ledger is `0`, not `NULL` |
| `test_receive_then_ship_happy_path` | the arithmetic |
| `test_overdraft_raises_with_the_numbers` | invariant 1 + the 6.2 exception contract |
| `test_failed_order_is_fully_rolled_back` | invariant 3 — **the atomicity test** |
| `test_every_write_leaves_an_audit_row` | invariant 2 — ledger completeness |
| `test_receive_with_client_token_is_idempotent` | the 18.3 replay guarantee |

The tests import `inventory.py` — the module extracted from this notebook — and
run under pytest in a subprocess (**15.4**'s temp-project pattern).

In [ ]:
def put(name: str, source: str) -> None:
    (PROJ / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")


put("conftest.py", """
    import pytest

    from inventory import InventoryRepo, connect, init_db


    @pytest.fixture
    def repo():
        # A fresh, fully isolated warehouse per test (10.3 + 15.4)
        conn = connect(":memory:")
        init_db(conn)
        r = InventoryRepo(conn)
        r.add_product("CBL-3M", "3 m HDMI cable", reorder_level=10)
        r.add_product("SSD-1TB", "1 TB NVMe drive", reorder_level=4)
        yield r
        conn.close()
""")

put("test_inventory.py", """
    import pytest

    from inventory import InsufficientStockError


    def movement_count(repo):
        return repo.conn.execute(
            "SELECT COUNT(*) AS n FROM stock_movements").fetchone()["n"]


    def test_new_product_has_zero_stock(repo):
        assert repo.stock_level("CBL-3M") == 0


    def test_receive_then_ship_happy_path(repo):
        repo.receive("CBL-3M", 40, reason="PO-1")
        repo.ship_order("SO-1", {"CBL-3M": 15})
        assert repo.stock_level("CBL-3M") == 25


    def test_overdraft_raises_with_the_numbers(repo):
        repo.receive("CBL-3M", 3, reason="PO-1")
        with pytest.raises(InsufficientStockError) as excinfo:
            repo.ship_order("SO-1", {"CBL-3M": 5})
        err = excinfo.value
        assert (err.sku, err.requested, err.available) == ("CBL-3M", 5, 3)


    def test_failed_order_is_fully_rolled_back(repo):
        # Invariant 3: a mid-order failure leaves NOTHING behind.
        repo.receive("CBL-3M", 40, reason="PO-1")
        repo.receive("SSD-1TB", 2, reason="PO-1")
        before = (repo.stock_level("CBL-3M"), repo.stock_level("SSD-1TB"),
                  movement_count(repo))

        with pytest.raises(InsufficientStockError):
            # line 1 (CBL-3M) INSERTs inside the transaction, line 2 then fails
            repo.ship_order("SO-1", {"CBL-3M": 5, "SSD-1TB": 99})

        after = (repo.stock_level("CBL-3M"), repo.stock_level("SSD-1TB"),
                 movement_count(repo))
        assert after == before == (40, 2, 2)


    def test_every_write_leaves_an_audit_row(repo):
        # Invariant 2: the ledger records every change, and sums to the level.
        repo.receive("CBL-3M", 40, reason="PO-1")
        repo.ship_order("SO-1", {"CBL-3M": 5})
        repo.adjust("CBL-3M", -2, reason="damaged in racking")
        history = repo.history("CBL-3M")
        assert [m.quantity for m in history] == [40, -5, -2]
        assert sum(m.quantity for m in history) == repo.stock_level("CBL-3M") == 33


    def test_receive_with_client_token_is_idempotent(repo):
        first = repo.receive("CBL-3M", 10, reason="PO-9", client_token="PO-9/d1")
        replay = repo.receive("CBL-3M", 10, reason="PO-9", client_token="PO-9/d1")
        assert replay.id == first.id
        assert repo.stock_level("CBL-3M") == 10
        assert movement_count(repo) == 1
""")


def pytest_in(*args: str) -> str:
    """Run pytest in the extracted project (15.4 pattern)."""
    done = subprocess.run(
        [sys.executable, "-m", "pytest", "--no-header",
         "-p", "no:cacheprovider", *args],
        cwd=PROJ, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    return (f"$ pytest {' '.join(args)}\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip() + "\n" + "-" * 70
            + f"\nexit code: {done.returncode}")


print(pytest_in("-v"))

Read `test_failed_order_is_fully_rolled_back` again — it is this capstone in
one test. Receive stock, fire an order whose second line must fail, then assert
the *complete* state — both stock levels **and the movement count** — matches
what stood before. Asserting the count matters: a rollback that left a stray
audit row behind would pass a levels-only check and still be corruption.

---

## Common Mistakes & Pitfalls

- 🔴 **Building SQL with f-strings.** The injection demo is the reason. `?`
  placeholders, every time (**10.3**).
- 🔴 **Forgetting `PRAGMA foreign_keys = ON`.** It is per *connection*, not per
  database — bake it into a `connect()` helper so it cannot be forgotten.
- 🔴 **Helpers that commit.** A nested `with conn:` inside a private method
  commits the outer transaction's half-done work. One public method, one
  transaction.
- 🔴 **Check-then-write outside a transaction.** Reading `available`, deciding,
  then inserting in autocommit mode leaves a window where an interleaved writer
  overdraws the stock. The check must live *inside* the same transaction as the
  write. ⚠️ Nuance: with `sqlite3`'s default (legacy) transaction control, `with conn:`
  does not `BEGIN` until the first *write* statement, so a read made before the first
  write still runs in autocommit. Single-connection demos like this notebook are safe;
  under real concurrency open the transaction explicitly — `BEGIN IMMEDIATE`, or the
  3.12+ `autocommit=False` connection mode (extension exercise 4).
- 🔴 **`UPDATE`-ing the ledger.** Corrections are new `adjust` movements. A
  ledger you can edit in place is not an audit trail.
- 🔴 **A denormalised total with no reconciliation.** Cache if you must — but
  schedule the reconciliation query, or the cache is a slow-acting bug.
- ⚠️ **Comparing SQLite versions as strings.** Gate features on
  `sqlite3.sqlite_version_info`, the tuple.
- ⚠️ **`REAL` for quantities.** Count in integers (units, grams, cents) —
  floating-point stock drifts one rounding error at a time.

## Best Practices

- Inject the connection; construct it in exactly one `connect()` helper that
  sets `row_factory` and pragmas, so no connection is ever half-configured.
- Let the schema enforce what it can (`CHECK`, `UNIQUE`, foreign keys) and the
  repo enforce what it cannot — a non-negative *total* needs `SUM`, which no
  row-level `CHECK` can see.
- Raise domain exceptions carrying data (**6.2**), chained with `from exc` so
  the `sqlite3` cause survives for debugging.
- Return frozen dataclasses (**5.3**) — callers get names and immutability, not
  tuple indices.
- Make externally triggered writes idempotent: client tokens backed by a
  `UNIQUE` column (**18.3**).
- Keep service logic behind a `Protocol` (**16.4**): test it against a
  five-line fake, type-check it against the real repo.
- Test the invariants, not the implementation — atomicity and audit
  completeness survive refactors; "calls `execute()` three times" does not.

## Extension exercises

1. **Reservations.** Add a `reservations` table: an order first *reserves*
   stock, shipping consumes the reservation. New invariant:
   `SUM(movements) − SUM(active reservations) ≥ 0`. Decide which operations
   must share a transaction.
2. **Enforce invariant 1 in the schema.** Write a `CREATE TRIGGER` that
   rejects any movement driving a product's `SUM(quantity)` below zero. What
   does the trigger cost per write, and is the repo's Python check now
   redundant?
3. **Migration thinking.** Add a `location` column to `products` without
   losing data: a numbered migration script per change, tracked with
   `PRAGMA user_version`. Apply v1 → v2 against a copy of a populated database.
4. **Concurrency, for real.** Open two connections to one *file-backed*
   database and make simultaneous shipments collide. Investigate
   `BEGIN IMMEDIATE`, `busy_timeout` and WAL mode (**12** supplies the threads
   to provoke it).
5. **Port the repo to SQLAlchemy 2.0** (**10.4**): models for both tables,
   `Session.begin()` as the transaction boundary. Keep this pytest suite green
   — it should not care.
6. **A FastAPI layer** (**18**): `POST /orders` calling `ship_order`, mapping
   `InsufficientStockError` → `409 Conflict`, with an `Idempotency-Key` header
   flowing into `receive`'s client token (**18.3**).
7. **Property-based tests** (**15.6**): with `hypothesis`, throw random
   operation sequences at the repo and assert all three invariants after every
   step.

In [ ]:
# ---- tidy up: close the warehouse, remove the only thing we wrote to disk ----
conn.close()
shutil.rmtree(WORK, ignore_errors=True)
print("connection closed; scratch directory removed:", not WORK.exists())